In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🕵️ LIME Model Explanations for Fraud Detection\n",
    "## Week 2 - Model Explainability\n",
    "\n",
    "This notebook implements LIME (Local Interpretable Model-agnostic Explanations) to explain why our fraud detection model makes specific predictions."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Install required packages\n",
    "!pip install lime pandas scikit-learn matplotlib numpy"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.preprocessing import StandardScaler, LabelEncoder\n",
    "import lime\n",
    "import lime.lime_tabular\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "print(\"✅ Libraries imported\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load and Prepare Your Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load your processed data\n",
    "df = pd.read_csv('../data/processed/unified_claims.csv')\n",
    "print(f\"📊 Data shape: {df.shape}\")\n",
    "print(f\"📋 Columns: {df.columns.tolist()}\")\n",
    "df.head(3)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Data Preprocessing"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check what fraud-related columns we have\n",
    "fraud_columns = [col for col in df.columns if any(word in col.lower() for word in \n",
    "                ['fraud', 'suspect', 'abuse', 'investigation', 'flag', 'potential'])]\n",
    "print(f\"🎯 Fraud-related columns: {fraud_columns}\")\n",
    "\n",
    "# Create target variable\n",
    "if 'FRAUD_TYPE' in df.columns:\n",
    "    # If you have explicit fraud types\n",
    "    df['is_fraud'] = (df['FRAUD_TYPE'] != 'No Fraud').astype(int)\n",
    "    print(f\"🔍 Fraud distribution:\\n{df['FRAUD_TYPE'].value_counts()}\")\n",
    "elif 'PotentialFraud' in df.columns:\n",
    "    # If using Medicare data\n",
    "    df['is_fraud'] = (df['PotentialFraud'] == 'Yes').astype(int)\n",
    "    print(f\"🔍 Fraud distribution:\\n{df['PotentialFraud'].value_counts()}\")\n",
    "else:\n",
    "    # Fallback - create synthetic fraud labels for demo\n",
    "    print(\"⚠️ No fraud column found - creating synthetic labels for demo\")\n",
    "    np.random.seed(42)\n",
    "    df['is_fraud'] = (np.random.random(len(df)) < 0.08).astype(int)\n",
    "\n",
    "print(f\"\\n📈 Fraud rate: {df['is_fraud'].mean():.2%}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Select features for modeling\n",
    "# ADAPT THESE BASED ON YOUR ACTUAL COLUMNS!\n",
    "numeric_features = []\n",
    "categorical_features = []\n",
    "\n",
    "# Check what numeric columns we have\n",
    "numeric_candidates = ['AGE', 'Age', 'AMOUNT', 'Amount', 'BILLED', 'Billed']\n",
    "for col in df.columns:\n",
    "    if any(candidate in col for candidate in numeric_candidates) and df[col].dtype in ['int64', 'float64']:\n",
    "        numeric_features.append(col)\n",
    "    elif col not in ['is_fraud'] + fraud_columns and df[col].dtype == 'object':\n",
    "        categorical_features.append(col)\n",
    "\n",
    "print(f\"🔢 Numeric features: {numeric_features}\")\n",
    "print(f\"📝 Categorical features: {categorical_features}\")\n",
    "\n",
    "# If no features found, use some defaults\n",
    "if not numeric_features:\n",
    "    numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()\n",
    "    numeric_features = [f for f in numeric_features if f != 'is_fraud']\n",
    "    print(f\"🤖 Using auto-detected numeric features: {numeric_features[:5]}...\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Prepare features and target\n",
    "X = df[numeric_features].fillna(0)\n",
    "y = df['is_fraud']\n",
    "\n",
    "# If we have categorical features, encode them\n",
    "if categorical_features:\n",
    "    for col in categorical_features:\n",
    "        le = LabelEncoder()\n",
    "        X[col] = le.fit_transform(df[col].astype(str))\n",
    "\n",
    "print(f\"🎯 Final feature matrix shape: {X.shape}\")\n",
    "print(f\"📋 Features: {X.columns.tolist()}\")\n",
    "\n",
    "# Train-test split\n",
    "X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)\n",
    "print(f\"📚 Training set: {X_train.shape}, Test set: {X_test.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Train a Fraud Detection Model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Train Random Forest model\n",
    "model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')\n",
    "model.fit(X_train, y_train)\n",
    "\n",
    "# Evaluate model\n",
    "train_score = model.score(X_train, y_train)\n",
    "test_score = model.score(X_test, y_test)\n",
    "\n",
    "print(f\"📊 Model Performance:\")\n",
    "print(f\"   Training Accuracy: {train_score:.3f}\")\n",
    "print(f\"   Test Accuracy: {test_score:.3f}\")\n",
    "\n",
    "# Feature importance\n",
    "feature_importance = pd.DataFrame({\n",
    "    'feature': X.columns,\n",
    "    'importance': model.feature_importances_\n",
    "}).sort_values('importance', ascending=False)\n",
    "\n",
    "print(f\"\\n🔝 Top 5 Most Important Features:\")\n",
    "print(feature_importance.head())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Implement LIME Explanations"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize LIME Explainer\n",
    "explainer = lime.lime_tabular.LimeTabularExplainer(\n",
    "    training_data=X_train.values,\n",
    "    feature_names=X.columns.tolist(),\n",
    "    class_names=['Genuine', 'Fraud'],\n",
    "    mode='classification',\n",
    "    random_state=42\n",
    ")\n",
    "\n",
    "print(\"✅ LIME Explainer initialized\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Test LIME on a few instances\n",
    "def generate_lime_explanation(instance_idx, num_features=5):\n",
    "    \"\"\"Generate and display LIME explanation for a specific instance\"\"\"\n",
    "    instance = X_test.iloc[instance_idx:instance_idx+1].values[0]\n",
    "    true_label = y_test.iloc[instance_idx]\n",
    "    \n",
    "    # Generate explanation\n",
    "    explanation = explainer.explain_instance(\n",
    "        instance, \n",
    "        model.predict_proba, \n",
    "        num_features=num_features\n",
    "    )\n",
    "    \n",
    "    print(f\"\\n🔍 Instance {instance_idx} (True: {'Fraud' if true_label else 'Genuine'})\")\n",
    "    print(\"=\" * 50)\n",
    "    \n",
    "    # Show prediction probabilities\n",
    "    proba = model.predict_proba([instance])[0]\n",
    "    print(f\"📊 Prediction: {proba[1]:.1%} fraud probability\")\n",
    "    \n",
    "    # Get text explanation\n",
    "    exp_list = explanation.as_list()\n",
    "    print(f\"\\n🎯 Top {num_features} contributing factors:\")\n",
    "    for i, (feature, importance) in enumerate(exp_list[:num_features], 1):\n",
    "        print(f\"   {i}. {feature}: {importance:.3f}\")\n",
    "    \n",
    "    return explanation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate explanations for 3 different cases\n",
    "print(\"🚀 Generating LIME Explanations for Test Cases:\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "# Find some interesting cases\n",
    "fraud_cases = y_test[y_test == 1].index[:2]\n",
    "genuine_cases = y_test[y_test == 0].index[:1]\n",
    "\n",
    "test_cases = list(fraud_cases) + list(genuine_cases)\n",
    "\n",
    "for case_idx in test_cases:\n",
    "    explanation = generate_lime_explanation(case_idx)\n",
    "    \n",
    "    # Show visual explanation in notebook\n",
    "    print(\"\\n📈 Visual Explanation:\")\n",
    "    explanation.show_in_notebook()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Create Human-Readable Explanations"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def create_business_explanation(explanation, feature_names):\n",
    "    \"\"\"Convert LIME output to business-friendly explanations\"\"\"\n",
    "    exp_list = explanation.as_list()\n",
    "    \n",
    "    reasons = []\n",
    "    for feature, importance in exp_list[:3]:  # Top 3 reasons\n",
    "        # Parse LIME feature string\n",
    "        if ' < ' in feature or ' > ' in feature:\n",
    "            # Feature with threshold (e.g., \"Amount Billed < 5000\")\n",
    "            parts = feature.split(' ')\n",
    "            feature_name = ' '.join(parts[:-2])\n",
    "            operator = parts[-2]\n",
    "            value = parts[-1]\n",
    "            \n",
    "            if importance > 0:\n",
    "                if operator == '<':\n",
    "                    reason = f\"{feature_name} is lower than {value}\"\n",
    "                else:\n",
    "                    reason = f\"{feature_name} is higher than {value}\"\n",
    "            else:\n",
    "                if operator == '<':\n",
    "                    reason = f\"{feature_name} is higher than {value}\"\n",
    "                else:\n",
    "                    reason = f\"{feature_name} is lower than {value}\"\n",
    "        else:\n",
    "            # Simple feature importance\n",
    "            if importance > 0:\n",
    "                reason = f\"High {feature}\"\n",
    "            else:\n",
    "                reason = f\"Low {feature}\"\n",
    "        \n",
    "        reasons.append(reason)\n",
    "    \n",
    "    return reasons\n",
    "\n",
    "# Test business explanations\n",
    "print(\"💼 Business-Friendly Explanations:\")\n",
    "print(\"=\" * 50)\n",
    "\n",
    "for case_idx in test_cases[:1]:  # Just show first case\n",
    "    instance = X_test.iloc[case_idx:case_idx+1].values[0]\n",
    "    explanation = explainer.explain_instance(instance, model.predict_proba, num_features=5)\n",
    "    \n",
    "    business_reasons = create_business_explanation(explanation, X.columns.tolist())\n",
    "    \n",
    "    print(f\"\\n📋 Claim Analysis:\")\n",
    "    for i, reason in enumerate(business_reasons, 1):\n",
    "        print(f\"   {i}. {reason}\")\n",
    "    \n",
    "    # Add action recommendation\n",
    "    proba = model.predict_proba([instance])[0][1]\n",
    "    if proba > 0.7:\n",
    "        print(f\"\\n🚨 Recommendation: Flag for manual review (High risk: {proba:.1%})\")\n",
    "    elif proba > 0.3:\n",
    "        print(f\"\\n⚠️  Recommendation: Additional verification needed (Medium risk: {proba:.1%})\")\n",
    "    else:\n",
    "        print(f\"\\n✅ Recommendation: Approve automatically (Low risk: {proba:.1%})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Save the LIME Explainer"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pickle\n",
    "\n",
    "# Save the model and explainer for later use\n",
    "model_data = {\n",
    "    'model': model,\n",
    "    'feature_names': X.columns.tolist(),\n",
    "    'explainer': explainer,\n",
    "    'preprocessor': None  # You can add your preprocessor here\n",
    "}\n",
    "\n",
    "with open('../models/model_with_lime.pkl', 'wb') as f:\n",
    "    pickle.dump(model_data, f)\n",
    "\n",
    "print(\"💾 Model and LIME explainer saved to: ../models/model_with_lime.pkl\")\n",
    "print(\"\\n🎉 Week 2 - LIME Implementation Complete!\")\n",
    "print(\"Next: Integrate LIME explanations into your API and frontend\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}